### Langchain入门

In [3]:
import os
from statistics import multimode

# 1.加载环境变量
from dotenv import load_dotenv
from tornado.gen import multi

load_dotenv()
from langchain.tools import tool
from langchain.agents import *
# 2.定义工具，基础版，通过注释描述工具
@tool
def getWeather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"


# 3.定义Agent
agent = create_agent(
    "deepseek-chat", # 模型名称（必须是LangChain支持的模型）
    tools=[getWeather] # 工具集
)

# 4.调用模型
print("🚀 正在调用大模型...")
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "杭州今天天气如何?"}
    ]
})

# 5.打印结果
print(response)

🚀 正在调用大模型...


APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}

## 初始化模型

In [ ]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [4]:
response = model.invoke([
    {
        "role": "system",
        "content": "你是火箭队的武藏"
    },
    {
        "role": "user",
        "content": "请你自我介绍"
    }
])
print(response.content)

NameError: name 'model' is not defined

In [ ]:
stream = model.stream("你是谁？")
for chunk in stream:
    print(chunk.content,end="",flush=True)

## 创建Agent

In [ ]:
agent = create_agent(model=model)

# response = agent.invoke("你是谁？")
response = agent.stream({
    "messages": [
        {
            "role": "system",
            "content": "你是一个 helpful的助手"
        },
        {
            "role": "user",
            "content": "你是谁？"
        }
    ]
},stream_mode="messages")
for token,metadata in response:
    print(token.content,end="",flush=True)

### 消息类型

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

# 创建Agent
agent = create_agent(model="deepseek-chat")

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？")
    ]
})

print(response)

### 多模态消息

In [11]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import os
load_dotenv()
model = init_chat_model(model="qwen3.5-plus-2026-02-15",
                        model_provider="openai",
                        api_key=os.getenv("DASHSCOPE_API_KEY"),
                        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
agent = create_agent(model=model)

multi_message = HumanMessage(content=[
    {
        "type":"image",
        "url":"https://cdn.nlark.com/yuque/0/2026/png/64933360/1776757111973-7a3ed782-9e48-4ea9-8df1-ac449c4a25bd.png?x-oss-process=image%2Fformat%2Cwebp"
    },
    {
        "type":"text",
        "text":"这个图片描述了什么？"
    }
])
#flush是Python中的一个参数，用于控制是否立即输出，还是等待缓冲区满后再输出
for token,metadata in agent.stream(
    {"messages": [multi_message]},
    stream_mode="messages"):
    print(token.content,end="",flush=True)



这张图片是一张**序列图（Sequence Diagram）**，详细描述了一个 **AI Agent（智能体）如何通过调用外部工具（Function Calling / Tool Use）来回答用户关于天气的提问** 的完整技术流程。

以下是图中展示的具体步骤和交互逻辑：

**1. 主要参与者（从左到右）：**
*   **用户 (User)**：发起请求的人。
*   **Agent**：中间协调者，负责接收用户请求、与模型交互以及执行具体工具。
*   **模型 (Model)**：大语言模型（LLM），负责理解意图、决定是否需要调用工具以及生成最终回答。

**2. 详细流程解析：**

1.  **用户提问**：用户向 Agent 发送指令：“杭州天气如何？”
2.  **参数组装**：Agent 接收到问题后，将**用户问题 (message)** 和预设的**工具信息 (tools)**（图中黄色便签显示：工具名为 `get_weather`，描述为查询天气，参数为 `city`）组装成请求参数。
3.  **第一次调用模型**：Agent 将组装好的参数发送给模型。
4.  **模型判断**：模型接收请求，进行内部推理（右侧自循环箭头），判断用户意图是否需要调用工具。模型识别出需要调用 `get_weather` 工具，并提取出参数“杭州”。
5.  **返回工具调用指令**：模型告诉 Agent 需要调用工具（图中虚线箭头“【调用工具】”），并返回了具体的工具名称和参数（黄色便签：`name: get_weather`, `parameter: 杭州`）。
6.  **执行工具**：Agent 根据模型的指令，实际执行了天气查询工具（右侧自循环箭头“执行工具”）。
7.  **第二次调用模型（带结果）**：Agent 获取到工具的执行结果（例如：“The weather in 杭州 is sunny!”），并将这个结果再次发送给模型。
8.  **生成回答**：模型结合工具返回的具体数据，进行分析和润色（右侧自循环箭头“分析结果，生成回答”），生成自然语言的回复。
9.  **返回最终答案**：模型将生成的回答（“杭州天气晴朗！”）返回给 Agent。
10. **回复用户**：Agent 最终将答案“杭州天气晴朗！”发送给用户。

**总结：

### 本地图片需要先转成base64编码格式

In [12]:
from ipywidgets import FileUpload
from IPython.display import display
uploader = FileUpload(accept='*',multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [13]:
print(uploader.value)

({'name': '集装箱.png', 'type': 'image/png', 'size': 2741143, 'content': <memory at 0x116aadc00>, 'last_modified': datetime.datetime(2026, 4, 3, 8, 50, 54, 643000, tzinfo=datetime.timezone.utc)},)


In [15]:
import base64
#获取第一个(也是唯一一个)上传的文件
uploaded_file =uploader.value[0]
# 获取其内存视图
content_mv = uploaded_file["content"]
# 转换内存视图->字节
img_bytes =bytes(content_mv)

In [16]:
import base64

# 例如，有一个用户上传的文件，是字节格式img_bytes，我们先将其进行base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

# 调用Agent，发送消息
response = agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
)
for token,metadata in response:
    print(token.content,end="",flush=True)


KeyboardInterrupt: 

# 提示词工程

添加身份，和特定的指令

In [18]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

school_name = "黑马程序员"

In [19]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

金曦城（Auroria）——悬浮于金星硫酸云层中的浮空都市群，依托碳纤维骨架与抗酸薄膜穹顶，核心能源来自大气层深处的超高温涡轮阵列，城市底部持续释放蓝色等离子光柱穿透云海。

### 结构化输出


In [ ]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response['messages'][-1].content)

### Langchain简化了结构化输出

In [ ]:
from pydantic import BaseModel
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [ ]:
# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

In [ ]:
city = response['structured_response']

# 工具类的定义

In [5]:
from langchain.tools import tool
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

### 可以通过pydantic来定义参数

In [2]:
from typing import Literal
from pydantic import Field, BaseModel

# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

In [4]:
# 定义一个查询天气的tool
from langchain.tools import tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [7]:
# 调用数学工具
tool1.invoke({"x": 467})

# 调用查询天气工具
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

In [16]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
load_dotenv()

agent = create_agent(
    model="deepseek-chat",
    tools=[tool1, get_weather]
)

In [18]:
for token,metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

我来帮您查询杭州接下来几天的天气情况。Current weather in 杭州: 22 degrees C
Next 5 days: Sunny根据查询结果，杭州当前的天气是22摄氏度。接下来5天的天气预报显示都是晴天。

看起来天气信息比较简略，只有"Sunny"这个描述。如果您需要更详细的天气预报信息，比如具体的温度范围、降水概率、风力等，可能需要使用更详细的天气查询服务。

In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467的平方根是多少?")]}
)

In [21]:
for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467的平方根是多少?
None
================================== Ai Message ==================================

我来帮你计算467的平方根。
Tool Calls:
  square_root (call_00_XSU6Ravv57jRtpPfwKR6dENu)
 Call ID: call_00_XSU6Ravv57jRtpPfwKR6dENu
  Args:
    x: 467
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================== Ai Message ==================================

467的平方根大约是 **21.61018278497431**。

这是一个无理数，所以这个小数表示是近似值。
None


# 预定义工具-Tavily

In [34]:
from dotenv import load_dotenv
load_dotenv()

# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

# 初始化工具，并设置参数，具体参数设置参考官网
search_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [28]:
search_tool.invoke("奶龙是谁？")

{'query': '奶龙是谁？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://zh.zalora.com.hk/blog/lifestyle/who-is-nailoong/',
   'title': '什麼是奶龍？除了奶龍，還有朋友小七和反派暴暴龍！ - ZALORA 香港',
   'content': '奶龍（Nailoong）是一隻來自異星的幼龍（小恐龍設定），最大特色係 duang duang 的大肚子，外形呆萌可愛，但又帶啲小機靈。',
   'score': 0.9998832,
   'raw_content': None},
  {'url': 'https://baike.baidu.com/item/%E5%A5%B6%E9%BE%99/58798883',
   'title': '奶龙_百度百科',
   'content': '奶龙是动画《奶龙》系列及其衍生作品中的主角。它是一只拥有“duangduang”大肚子的异星幼龙，呆萌可爱又带点小机灵的大吃货一枚。来到地球遇到了活泼搞笑、富有正义感、',
   'score': 0.9997658,
   'raw_content': None},
  {'url': 'https://zh.wikipedia.org/wiki/%E5%A5%B6%E9%BE%99',
   'title': '奶龍- 維基百科，自由的百科全書',
   'content': '奶龍（英語：Nailoong），是《奶龍》系列動畫及其衍生作品中的虛構恐龍，由深圳第七印象文化傳媒有限公司設計。 奶龍. 穿深圳校服的奶龍塑像. 創作者, 謝承恩.',
   'score': 0.99943846,
   'raw_content': None},
  {'url': 'https://hachimi.fandom.com/zh/wiki/%E5%A5%B6%E9%BE%99',
   'title': '奶龙| 哈基米Wiki - Fandom',
   'content': '奶龙是由第七印象公司制作的动画系列《奶龙》的主角，是一只来自异星的小恐龙，由于所在的星球发生变故

In [29]:
agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="你是一个助手，需要帮助时可以使用工具解决问题。"
)

In [30]:
response = agent.invoke(
    {"messages": [HumanMessage(content="奶龙是谁？")]}
)

In [31]:
for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

奶龙是谁？
None
================================== Ai Message ==================================

我来帮你搜索一下关于"奶龙"的信息。
Tool Calls:
  tavily_search (call_00_S8dOnLlhflHRcALGz0tnT8Db)
 Call ID: call_00_S8dOnLlhflHRcALGz0tnT8Db
  Args:
    query: 奶龙是谁 角色介绍
None
================================= Tool Message =================================
Name: tavily_search

{"query": "奶龙是谁 角色介绍", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://baike.baidu.com/item/%E5%A5%B6%E9%BE%99/58798883", "title": "奶龙_百度百科", "content": "奶龙是动画《奶龙》系列及其衍生作品中的主角。它是一只拥有“duangduang”大肚子的异星幼龙，呆萌可爱又带点小机灵的大吃货一枚。来到地球遇到了活泼搞笑、富有正义感、", "score": 0.79484755, "raw_content": null}, {"url": "https://m.baike.com/wikiid/7270402341705957413", "title": "奶龙 - 抖音百科", "content": "奶龙，原创动画短片《小七和奶龙》中的主角，是一条来自异星的幼龙，颜色是黄色，圆圆的大脑袋和胖嘟嘟的身材，形象呆萌可爱，其表情包在微信等社交平台中传播，在动画中是小七最", "score": 0.7089592, "raw_content": null}, {"url": "https://www

### 自定义Tavily工具

In [54]:
from dotenv import load_dotenv
load_dotenv()
from langchain.tools import tool

from langchain_tavily import TavilySearch
# 使用tavily作为web搜索工具
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

In [55]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='"蒸蚌"是一个源自抖音的流行网络梗，主要有以下含义和来源：\n\n**1. 梗的来源：**\n- 起源于抖音博主训练三花猫分辨胡萝卜和纸巾的视频挑战\n- 博主用夸张高音喊"真棒"鼓励猫咪，被网友听成"蒸蚌"\n- 配合猫咪懵懂的表情形成魔性笑点，迅速走红网络\n\n**2. 含义解释：**\n- "蒸蚌"是"真棒"的谐音空耳（听错音）\n- 在猫咪挑战中，当猫咪蒙对答案时，主人会高喊"蒸蚌"（真棒）表示鼓励\n- 后来演变成一种表达"很棒"、"做得好"的幽默说法\n\n**3. 流行发展：**\n- 歌手周深在直播中接梗使用"蒸蚌"，进一步推动了该梗的传播\n- 形成了"萝卜纸巾猫"挑战，相关话题累计阅读量超18亿\n- 成为2026年初的全民级热梗，被广泛用于各种搞笑视频和表情包\n\n**4. 使用场景：**\n- 当某人做对事情或表现好时，用"蒸蚌"表示夸奖\n- 在搞笑视频中模仿猫咪挑战的魔性效果\n- 作为网络用语表达幽默的赞赏\n\n这个梗的核心笑点在于猫咪懵懂的表情与主人夸张的"蒸蚌"鼓励声形成的反差萌，加上谐音的空耳效果，形成了独特的幽默感。' reference=[Reference(title='周深直播接梗蒸蚌_新浪新闻', url='https://www.sina.cn/news/detail/5247857229563928.html'), Reference(title='蒸蚌！全网最火小猫咪喜提全球代言 - SocialBeta', url='https://socialbeta.com/article/110971'), Reference(title='现在互联网上最火的猫，从“耄耋”变成了“萝卜纸巾猫” - 搜狐', url='https://www.sohu.com/a/972125782_258858')]


# AgentMemory

### 短期记忆（InMemorySaver）

In [1]:
# langchain提供的checkpointer的默认实现，基于内存存储
from langgraph.checkpoint.memory import InMemorySaver

set checkpointer to InMemorySaver

In [3]:
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()
# 创建智能体时指定checkpointer，LangChain会自动帮我们管理历史会话记忆
agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()
)

发起调用时，需要指定thread_id，作为会话标识

In [4]:
from langchain.messages import HumanMessage

# 设定thread_id，作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫管哥，我最喜欢猫猫。")]},
    config # 调用时添加thread_id，区分不同会话
)

print(response)

{'messages': [HumanMessage(content='你好，我叫管哥，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='65c48fa3-e047-4283-929d-a09a3cc85aa2'), AIMessage(content='你好，管哥！🐱 听名字就感觉你是个特别有爱心的人——喜欢猫猫的人，心里一定住着温暖的小太阳！你家里有养猫吗？还是说现在暂时是“云养猫”爱好者？如果你有自家猫主子的照片或者故事，欢迎随时分享，我想它一定超级可爱！😸', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 15, 'total_tokens': 88, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 15}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'ae78f050-fdae-4c8c-ba25-d3be95ca47bf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbd5d-cfa9-7cf0-885a-708b59ec541a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 73, 'total_tokens': 88, 'input_token_details': {'cache_re

In [5]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢什么动物？")]},
    config # 调用时添加thread_id，区分不同会话
)
print(response)


{'messages': [HumanMessage(content='你好，我叫管哥，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='65c48fa3-e047-4283-929d-a09a3cc85aa2'), AIMessage(content='你好，管哥！🐱 听名字就感觉你是个特别有爱心的人——喜欢猫猫的人，心里一定住着温暖的小太阳！你家里有养猫吗？还是说现在暂时是“云养猫”爱好者？如果你有自家猫主子的照片或者故事，欢迎随时分享，我想它一定超级可爱！😸', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 15, 'total_tokens': 88, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 15}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'ae78f050-fdae-4c8c-ba25-d3be95ca47bf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbd5d-cfa9-7cf0-885a-708b59ec541a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 73, 'total_tokens': 88, 'input_token_details': {'cache_re